# Kansas Soil Water Atlas: weekly uint8 FAW cube

One pass over the daily hindcast maps in `faw/` (2002-01-01 onward). Writes to `data/` (local analysis files; the atlas page is made from them by `atlas.ipynb`):

| file | content |
|---|---|
| `faw_weekly_uint8.npy` | weekly mean FAW × 100, uint8 (weeks, 336, 832); 255 = outside Kansas |
| `weeks.csv` | year, week, first and last day, number of days of each layer |
| `statewide_daily.csv` | area-weighted statewide statistics of every day (percentiles, area in FAW classes, deficit/surplus, stored PAW) |
| `faw_days_yearly.npz` | per-pixel deficit-days and surplus-days of each calendar year, from the daily maps |

**Weeks** are fixed 7-day periods starting 1 January (week 1 = 1–7 Jan, ...); week 52 also takes the last 1–2 days of the year, so every year has exactly 52 weeks and the week × year heatmaps line up. Only complete weeks are written.

**Deficit-days / surplus-days** use the 0.5 reference, as in the FAW-days paper, but per calendar year (Jan–Dec) so they match the week × year layout.

In [ ]:
import os, sys, gzip, json, time
ROOT = r'D:\OneDrive - Kansas State University\Research projects\kansas-1km'   # the model output (faw/, soil/, boundaries/) is read from here
SEC  = r'D:\OneDrive - Kansas State University\Research projects\kansas-soil-water-atlas'   # this repository: page, figures, code and data
FIGS, DATA = os.path.join(SEC, 'figures'), os.path.join(SEC, 'data')
os.chdir(ROOT); sys.path.insert(0, ROOT)
import numpy as np, pandas as pd, rasterio

START = '2002-01-01'                   # PRISM CAI+radar era
REF   = 0.5                            # FAW-days reference
NODATA = 255

In [ ]:
last = sorted(f for f in os.listdir('faw') if f.startswith('faw_') and f.endswith('.tif'))[-1]
dates = pd.date_range(START, pd.Timestamp(last[4:12]))
missing = [d for d in dates if not os.path.exists(f'faw/faw_{d:%Y%m%d}.tif')]
assert not missing, f'missing days: {missing[:5]}'

with rasterio.open(f'faw/faw_{dates[-1]:%Y%m%d}.tif') as src:
    profile, shape, transform = src.profile, src.shape, src.transform
    valid = ~src.read(1, masked=True).mask
npix = int(valid.sum())
lat = transform.f + transform.e * (np.arange(shape[0]) + 0.5)
area = np.cos(np.deg2rad(np.broadcast_to(lat[:, None], shape)))[valid] * (abs(transform.a) * 111.32) * (abs(transform.e) * 111.32)  # km2
with rasterio.open('soil/taw.tif') as src:
    taw = src.read(1)[valid]               # mm, 0-60 cm

# fixed 52 weeks per year
doy  = dates.dayofyear.to_numpy()
week = np.minimum((doy - 1) // 7 + 1, 52)
key  = dates.year.to_numpy() * 100 + week
wk = (pd.DataFrame({'date': dates, 'key': key}).groupby('key').date.agg(['first', 'last', 'size']).reset_index())
wk['year'], wk['week'] = wk.key // 100, wk.key % 100
full = (wk.week < 52) & (wk['size'] == 7) | (wk.week == 52) & (wk['last'].dt.month == 12) & (wk['last'].dt.day == 31)
wk = wk[full].reset_index(drop=True)
print(f'{len(dates)} days, {dates[0]:%Y-%m-%d} to {dates[-1]:%Y-%m-%d}; {len(wk)} complete weeks; {npix} pixels, {area.sum():,.0f} km2, '
      f'PAW capacity {np.sum(taw * area) * 1e-6:.1f} km3')

In [ ]:
def wpct(v, w, q):
    "Area-weighted percentiles."
    o = np.argsort(v); c = np.cumsum(w[o]); c /= c[-1]
    return np.interp(q, c, v[o])

Q = np.array([0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95])
EDGES = np.linspace(0, 1, 11)                          # FAW classes 0-0.1, ..., 0.9-1.0
wa = area / area.sum()
years = np.unique(dates.year)
D = np.zeros((len(years), npix), 'float32'); S = np.zeros_like(D); N = np.zeros(len(years), int)
cube = np.lib.format.open_memmap(os.path.join(DATA, 'faw_weekly_uint8.npy.part'), mode='w+', dtype='uint8', shape=(len(wk),) + shape)
pos = {k: i for i, k in enumerate(wk.key)}
acc, nacc, cur, rows, t0 = np.zeros(npix, 'float64'), 0, None, [], time.time()

for t, d in enumerate(dates):
    with rasterio.open(f'faw/faw_{d:%Y%m%d}.tif') as src:
        f = np.clip(src.read(1)[valid].astype('float32'), 0, 1)
    assert np.isfinite(f).all() and (f > -1).all(), d
    k = key[t]
    if k != cur:
        acc[:], nacc, cur = 0, 0, k
    acc += f; nacc += 1
    if k in pos and nacc == wk['size'][pos[k]]:       # week complete: write the layer
        g = np.full(shape, NODATA, 'uint8'); g[valid] = np.round(acc / nacc * 100).astype('uint8')
        cube[pos[k]] = g
    y = np.searchsorted(years, d.year)
    D[y] += np.clip(REF - f, 0, None); S[y] += np.clip(f - REF, 0, None); N[y] += 1
    row = dict(date=d, mean=float(f @ wa), **{f'p{int(q * 100):02d}': v for q, v in zip(Q, wpct(f, wa, Q))},
               deficit=float(np.clip(REF - f, 0, None) @ wa), surplus=float(np.clip(f - REF, 0, None) @ wa),
               area_below_ref=float((f < REF) @ wa), paw_km3=float((f * taw) @ area * 1e-6))
    h = np.histogram(f, EDGES, weights=wa)[0]          # share of the state in each FAW class
    row.update({f'c{i}': v for i, v in enumerate(h)})
    rows.append(row)
    if t % 500 == 0: print(f'{d:%Y-%m-%d}  {time.time() - t0:5.0f} s', flush=True)

cube.flush(); del cube
os.replace(os.path.join(DATA, 'faw_weekly_uint8.npy.part'), os.path.join(DATA, 'faw_weekly_uint8.npy'))
daily = pd.DataFrame(rows); daily.to_csv(os.path.join(DATA, 'statewide_daily.csv'), index=False, float_format='%.4f')
wk[['year', 'week', 'first', 'last', 'size']].rename(columns={'first': 'start', 'last': 'end', 'size': 'ndays'}).to_csv(os.path.join(DATA, 'weeks.csv'), index=False)
np.savez_compressed(os.path.join(DATA, 'faw_days_yearly.npz'), years=years, ndays=N, deficit=D, surplus=S, valid=valid)
print(f'done in {(time.time() - t0) / 60:.1f} min')

In [ ]:
cube = np.load(os.path.join(DATA, 'faw_weekly_uint8.npy'), mmap_mode='r')
print(f'cube {cube.shape}, {cube.nbytes / 1e6:.0f} MB')

In [ ]:
# sanity check: one weekly layer against the daily maps it averages
i = len(wk) // 2
days = pd.date_range(wk['first'][i], wk['last'][i])
ref = np.mean([rasterio.open(f'faw/faw_{d:%Y%m%d}.tif').read(1)[valid] for d in days], 0)
diff = np.abs(cube[i][valid] / 100 - np.clip(ref, 0, 1))
print(f"week {wk.year[i]}-{wk.week[i]:02d}: max |uint8 - float| = {diff.max():.4f} (rounding limit 0.005)")